In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [4]:
DATA_PATH = Path("../data/raw/train.csv")

df = pd.read_csv(DATA_PATH)

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [5]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [6]:
def evaluate_model(name, pipeline):
    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    return {
        "Model": name,
        "RMSE": round(rmse, 2),
        "MAE": round(mae, 2),
        "R2 Score": round(r2, 4),
        "Pipeline": pipeline,
    }

In [7]:
linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_result = evaluate_model(
    "Linear Regression",
    linear_pipeline,
)

In [8]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

rf_result = evaluate_model(
    "Random Forest",
    rf_pipeline,
)

In [9]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )),
    ]
)

xgb_result = evaluate_model(
    "XGBoost",
    xgb_pipeline,
)

In [10]:
results = pd.DataFrame([
    linear_result,
    rf_result,
    xgb_result,
])

results = results.drop(columns=["Pipeline"])

results.sort_values(
    by="R2 Score",
    ascending=False,
)

,Model,RMSE,MAE,R2 Score
2,XGBoost,25576.20,15805.78,0.9147
1,Random Forest,28496.79,17509.56,0.8941
0,Linear Regression,29476.09,18284.67,0.8867


In [11]:
MODELS_DIR = Path("../models")

MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(
    xgb_pipeline,
    MODELS_DIR / "house_price_model.joblib"
)

print("Model saved successfully!")

Model saved successfully!


In [12]:
sample = X_test.iloc[[0]]

prediction = xgb_pipeline.predict(sample)

print(f"Predicted Price: ${prediction[0]:,.2f}")
print(f"Actual Price: ${y_test.iloc[0]:,.2f}")

Predicted Price: $139,763.06
Actual Price: $154,500.00
